In [3]:
import json
import numpy as np
from tqdm import tqdm
import evaluate
from rouge_score import rouge_scorer
from bert_score import BERTScorer

/home/xiaotang/miniconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
pred_path = "results/xsum-qwen2.5-7b-instruct-llm-attr.json"
predictions = load_predicitons(pred_path)

rouge_individual, rouge_average = compute_rouge_scores(predictions)

results = {
    'average_scores': {
        'rouge': rouge_average
    },
    'individual_scores': []
}

bert_individual, bert_average = compute_bertscore(predictions, batch_size=8)
# Add BERTScore averages to results
results['average_scores']['bertscore'] = bert_average
        
# Merge individual scores
results['individual_scores'] = merge_scores_by_id(rouge_individual, bert_individual)

# Save results
output_path = "results/metrics.json"
print(f"Saving results to {output_path}")
with open(output_path, 'w') as f:
    json.dump(results, f, indent=4)

# Print summary statistics
print("\nAverage Scores:")
print(f"Number of samples: {rouge_average['num_samples']}")

print("\nROUGE Scores:")
print(f"  ROUGE-1 F1: {rouge_average['rouge1_f1']:.4f}")
print(f"  ROUGE-2 F1: {rouge_average['rouge2_f1']:.4f}")
print(f"  ROUGE-L F1: {rouge_average['rougeL_f1']:.4f}")

print("\nBERTScore:")
print(f"  Precision: {bert_average['bertscore_precision']:.4f}")
print(f"  Recall: {bert_average['bertscore_recall']:.4f}")
print(f"  F1: {bert_average['bertscore_f1']:.4f}")

# print(rouge_individual)
# print(rouge_average)

Computing ROUGE scores: 100%|██████████| 3000/3000 [00:00<00:00, 3187.93it/s]


Initializing BERTScorer...


Computing BERTScore: 100%|██████████| 375/375 [00:16<00:00, 22.48it/s]


Saving results to results/metrics.json

Average Scores:
Number of samples: 3000

ROUGE Scores:
  ROUGE-1 F1: 0.2778
  ROUGE-2 F1: 0.0718
  ROUGE-L F1: 0.2012

BERTScore:
  Precision: 0.6711
  Recall: 0.7073
  F1: 0.6879


In [5]:
def load_predicitons(file_path):
    """Load prediciton results from a JSON file."""
    with open(file_path, 'r') as f:
        predictions = json.load(f)
    
    return predictions

def compute_rouge_scores(predictions):
    """Compute ROUGE scores for each prediction and return both individual and average scores."""
    # Initialize ROUGE scorer with metrics to compute
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    individual_scores = []
    
    # Keep track of all scores for calculating averages
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []
    
    for pred in tqdm(predictions, desc="Computing ROUGE scores"):
        reference = pred['reference_summary']
        generated = pred['generated_summary']
        
        # Skip if summaries are empty
        if not reference or not generated:
            continue
            
        scores = scorer.score(reference, generated)
        
        # Extract F1 scores for this sample
        sample_scores = {
            'id': pred['id'],
            'rouge1_f1': scores['rouge1'].fmeasure,
            'rouge2_f1': scores['rouge2'].fmeasure,
            'rougeL_f1': scores['rougeL'].fmeasure
        }
        
        # Add to individual scores list
        individual_scores.append(sample_scores)
        
        # Add to aggregation lists
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)
    
    # Calculate average scores
    average_scores = {
        'rouge1_f1': float(np.mean(rouge1_scores)),
        'rouge2_f1': float(np.mean(rouge2_scores)),
        'rougeL_f1': float(np.mean(rougeL_scores)),
        'num_samples': len(rouge1_scores)
    }
    
    return individual_scores, average_scores

def compute_bertscore(predictions, batch_size=8):
    """Compute BERTScore for each prediction and return both individual and average scores."""
    # Initialize BERTScorer
    print("Initializing BERTScorer...")
    bert_score = evaluate.load('bertscore')
    # scorer = BERTScorer(lang="en", rescale_with_baseline=True)
    
    # Prepare references, candidates, and IDs
    references = []
    candidates = []
    ids = []
    
    for pred in predictions:
        if pred['reference_summary'] and pred['generated_summary']:
            references.append(pred['reference_summary'])
            candidates.append(pred['generated_summary'])
            ids.append(pred['id'])
    
    # Initialize results
    individual_scores = []
    
    # Aggregation lists for average calculation
    all_precision = []
    all_recall = []
    all_f1 = []
    
    # Compute scores in batches to avoid memory issues
    for i in tqdm(range(0, len(references), batch_size), desc="Computing BERTScore"):
        batch_refs = references[i:i+batch_size]
        batch_cands = candidates[i:i+batch_size]
        batch_ids = ids[i:i+batch_size]
        
        # Calculate scores for this batch
        bert_score_res = bert_score.compute(predictions=batch_cands, 
                                            references=batch_refs,
                                            model_type="microsoft/deberta-xlarge-mnli", lang="en")
        P, R, F1 = bert_score_res['precision'], bert_score_res['recall'], bert_score_res['f1']
        # P, R, F1 = scorer.score(batch_cands, batch_refs)
        
        # Save individual scores
        for j in range(len(batch_ids)):
            sample_score = {
                'id': batch_ids[j],
                'bertscore_precision': float(P[j]),
                'bertscore_recall': float(R[j]),
                'bertscore_f1': float(F1[j])
            }
            individual_scores.append(sample_score)
            
            # Add to aggregation lists
            all_precision.append(float(P[j]))
            all_recall.append(float(R[j]))
            all_f1.append(float(F1[j]))
    
    # Calculate average scores
    average_scores = {
        'bertscore_precision': float(np.mean(all_precision)),
        'bertscore_recall': float(np.mean(all_recall)),
        'bertscore_f1': float(np.mean(all_f1)),
        'num_samples': len(individual_scores)
    }
    
    return individual_scores, average_scores

def merge_scores_by_id(rouge_scores, bert_scores):
    """Merge ROUGE and BERTScore results by ID."""
    # Create dictionaries for fast lookup
    rouge_dict = {score['id']: score for score in rouge_scores}
    bert_dict = {score['id']: score for score in bert_scores}
    
    # Get all unique IDs
    all_ids = set(rouge_dict.keys()) | set(bert_dict.keys())
    
    # Merge scores
    merged_scores = []
    for id in all_ids:
        merged = {'id': id}
        
        # Add ROUGE scores if available
        if id in rouge_dict:
            for k, v in rouge_dict[id].items():
                if k != 'id':
                    merged[k] = v
        
        # Add BERTScore scores if available
        if id in bert_dict:
            for k, v in bert_dict[id].items():
                if k != 'id':
                    merged[k] = v
        
        merged_scores.append(merged)
    
    return merged_scores

In [10]:
attr_rouge_individual = rouge_individual 
attr_bert_individual = bert_individual
print(attr_rouge_individual)

[{'id': '38264402', 'rouge1_f1': 0.25, 'rouge2_f1': 0.04347826086956522, 'rougeL_f1': 0.20833333333333331}, {'id': '34227252', 'rouge1_f1': 0.6153846153846153, 'rouge2_f1': 0.3243243243243243, 'rougeL_f1': 0.25641025641025644}, {'id': '38537698', 'rouge1_f1': 0.2962962962962963, 'rouge2_f1': 0.07692307692307691, 'rougeL_f1': 0.18518518518518517}, {'id': '36175342', 'rouge1_f1': 0.13636363636363635, 'rouge2_f1': 0.047619047619047616, 'rougeL_f1': 0.09090909090909091}, {'id': '39070183', 'rouge1_f1': 0.358974358974359, 'rouge2_f1': 0.05405405405405405, 'rougeL_f1': 0.15384615384615383}, {'id': '38899892', 'rouge1_f1': 0.3018867924528302, 'rouge2_f1': 0.0784313725490196, 'rougeL_f1': 0.18867924528301888}, {'id': '39339718', 'rouge1_f1': 0.3076923076923077, 'rouge2_f1': 0.08, 'rougeL_f1': 0.15384615384615385}, {'id': '34571446', 'rouge1_f1': 0.38596491228070173, 'rouge2_f1': 0.10909090909090909, 'rougeL_f1': 0.2105263157894737}, {'id': '36892983', 'rouge1_f1': 0.03773584905660377, 'rouge2_

In [8]:
base_rouge_individual = rouge_individual
base_bert_individual = bert_individual
print(base_rouge_individual)

[{'id': '38264402', 'rouge1_f1': 0.08163265306122448, 'rouge2_f1': 0.0, 'rougeL_f1': 0.08163265306122448}, {'id': '34227252', 'rouge1_f1': 0.48648648648648646, 'rouge2_f1': 0.17142857142857143, 'rougeL_f1': 0.2162162162162162}, {'id': '38537698', 'rouge1_f1': 0.24137931034482762, 'rouge2_f1': 0.03571428571428571, 'rougeL_f1': 0.17241379310344826}, {'id': '36175342', 'rouge1_f1': 0.14285714285714285, 'rouge2_f1': 0.05, 'rougeL_f1': 0.09523809523809525}, {'id': '39070183', 'rouge1_f1': 0.3333333333333333, 'rouge2_f1': 0.04347826086956522, 'rougeL_f1': 0.16666666666666666}, {'id': '38899892', 'rouge1_f1': 0.2692307692307692, 'rouge2_f1': 0.07999999999999999, 'rougeL_f1': 0.1923076923076923}, {'id': '39339718', 'rouge1_f1': 0.2916666666666667, 'rouge2_f1': 0.08695652173913043, 'rougeL_f1': 0.08333333333333333}, {'id': '34571446', 'rouge1_f1': 0.29850746268656714, 'rouge2_f1': 0.06153846153846155, 'rougeL_f1': 0.1791044776119403}, {'id': '36892983', 'rouge1_f1': 0.25396825396825395, 'rouge2

In [12]:
rouge_diff = find_largest_differences(base_rouge_individual, attr_rouge_individual)
bert_diff = find_largest_differences(base_bert_individual, attr_bert_individual)

print(rouge_diff)

[{'id': '36989435', 'method1_score': 0.1142857142857143, 'method2_score': 0.41379310344827586, 'abs_difference': 0.2995073891625616, 'raw_difference': 0.2995073891625616}, {'id': '37801034', 'method1_score': 0.44827586206896547, 'method2_score': 0.15384615384615385, 'abs_difference': 0.2944297082228116, 'raw_difference': -0.2944297082228116}, {'id': '25943093', 'method1_score': 0.36000000000000004, 'method2_score': 0.06666666666666667, 'abs_difference': 0.2933333333333334, 'raw_difference': -0.2933333333333334}, {'id': '36016648', 'method1_score': 0.08333333333333333, 'method2_score': 0.37499999999999994, 'abs_difference': 0.29166666666666663, 'raw_difference': 0.29166666666666663}, {'id': '38949497', 'method1_score': 0.45454545454545453, 'method2_score': 0.1818181818181818, 'abs_difference': 0.2727272727272727, 'raw_difference': -0.2727272727272727}, {'id': '27412906', 'method1_score': 0.17857142857142858, 'method2_score': 0.4444444444444444, 'abs_difference': 0.2658730158730158, 'raw

In [11]:
from typing import List, Dict, Any

def find_largest_differences(scores1: List[Dict[str, Any]], 
                             scores2: List[Dict[str, Any]], 
                             metric: str = 'rougeL_f1',
                             top_k: int = 10) -> List[Dict[str, Any]]:
    """
    Find the top K samples with the largest absolute difference in ROUGE scores.
    
    Args:
        scores1: List of score dictionaries from first method
        scores2: List of score dictionaries from second method
        metric: The ROUGE metric to compare (default: 'rougeL_f1')
        top_k: Number of top differences to return
        
    Returns:
        List of dictionaries with difference information
    """
    # Create dictionaries for quick lookup
    scores1_dict = {item['id']: item for item in scores1}
    scores2_dict = {item['id']: item for item in scores2}
    
    # Find common IDs
    common_ids = set(scores1_dict.keys()) & set(scores2_dict.keys())
    
    # Calculate differences
    differences = []
    for id in common_ids:
        if metric in scores1_dict[id] and metric in scores2_dict[id]:
            score1 = scores1_dict[id][metric]
            score2 = scores2_dict[id][metric]
            abs_diff = abs(score1 - score2)
            raw_diff = score2 - score1  # Positive means method2 is better
            
            differences.append({
                'id': id,
                'method1_score': score1,
                'method2_score': score2,
                'abs_difference': abs_diff,
                'raw_difference': raw_diff
            })
    
    # Sort by absolute difference (largest first)
    differences.sort(key=lambda x: x['abs_difference'], reverse=True)
    
    # Return top K
    return differences[:top_k]

In [ ]:
def mean_score(scores):
    return sum(scores) / len(scores)

def eval_rouge_scores(preds, labels):
    rouge = evaluate.load('rouge')
    rouge_scores = rouge.compute(preditions=preds,
                                 references=labels)
    
    metrics = {
        'rouge1': rouge_scores['rouge1'],
        'rouge2': rouge_scores['rouge2'],
        'rougeLsum': rouge_scores['rougeLsum'],
    }

    return metrics

def eval_bert_scores(preds, labels):
    bert_score = evaluate.load('bertscore')
    bert_score_res = bert_score.compute(predictions=preds, 
                                        references=labels, 
                                        model_type="microsoft/deberta-xlarge-mnli", lang="en")
    metrics = {
        'bertscore_p': mean_score(bert_score_res['precision']),
        'bertscore_r': mean_score(bert_score_res['recall']),
        'bertscore_f1': mean_score(bert_score_res['f1']),
    }

    return metrics

def print_metrics(metrics):
    for metric_name, value in metrics.items():
        print(f"{metric_name}: {value:.4f}")